## <span style='color:green'>1.0 Arrival Data Frame </span>
#### Goals of the Project 
This is a Data cleaning project aimed at preparing Data for Machine learning 

In [1]:
import numpy  as np
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sns 
import dateutil.parser as parser 
from datetime import datetime
import dateutil.parser as parser 
from datetime import datetime
from sklearn.linear_model import LinearRegression #to build a linear regression
%config InlineBackend.figure_format = 'retina'

### Stored DF retrieved Arrival_DF 
- The use of the %store -r Arrival_DF is to retrive the Data Frame from the DELI MACHINE LEARNING PROJECT PART1

In [2]:
%store -r Arrival_DF
Arrival_DF.shape

(900164, 16)

In [3]:
# creation a corelation between these features and the pax load 
#pax load and extended week = 0.8 


In [4]:
Arrival_DF.head(1) 

,Date,Day of week,Weekend (0/1),Holiday (0/1),Festival,Overlap with Weekend,Extended weekend,boarded pax,Original Airport,Destination airport,Arrival Datetime,Seat Capacity,traffic_type,Flight Number,Terminal,Airline Code
0,2018-12-31,Monday,0,0,0,0,0,155,IXR,DEL,2018-12-31 00:15:00,186.0,D,G8144,T2,G8


In [5]:
Arrival_DF.columns # Looking at what columns we do have 

Index(['Date', 'Day of week', 'Weekend (0/1)', 'Holiday (0/1)', 'Festival',
       'Overlap with Weekend', 'Extended weekend', 'boarded pax',
       'Original Airport', 'Destination airport', 'Arrival Datetime',
       'Seat Capacity', 'traffic_type', 'Flight Number', 'Terminal',
       'Airline Code'],
      dtype='object')

In [6]:
Arrival_DF.dtypes # Checking out the available Data types 

Date                    datetime64[ns]
Day of week                     object
Weekend (0/1)                    int32
Holiday (0/1)                    int64
Festival                         int64
Overlap with Weekend             int32
Extended weekend                 int32
boarded pax                     object
Original Airport                object
Destination airport             object
Arrival Datetime                object
Seat Capacity                   object
traffic_type                    object
Flight Number                   object
Terminal                        object
Airline Code                    object
dtype: object

### 2. Converting Seat Capacity and boarded Pax into int for Pax ratio
- This section involves converting both the Seat Capacity and boarded Pax into an integer , in this way , a pax ratio will be calulated .
- We apply pd.to_numeric and downcasting it is converting a data type to a lower precision or a smaller range number.

In [7]:
Arrival_DF["Seat Capacity"]=Arrival_DF["Seat Capacity"].apply(pd.to_numeric,errors='coerce',downcast='integer')

In [8]:
Arrival_DF["boarded pax"]=Arrival_DF["boarded pax"].apply(pd.to_numeric,errors='coerce',downcast='integer')

### 3. Removing duplicated rows 
- This chapter aims at observing available duplicates and removing them 

In [9]:
duplicateRows=Arrival_DF[Arrival_DF.duplicated()]
duplicateRows.shape # Flight number and Flight date  time .. We do not have an duplicate values . 

(108, 16)

In [10]:
print("percentage of duplicates is :", (len(duplicateRows)/len(Arrival_DF)*100))
#only 0.056 percent and will be deleted .. too small to make any different 

percentage of duplicates is : 0.011997813731719998


In [11]:
Arrival_DF=Arrival_DF.drop_duplicates()
len(Arrival_DF) # meangless since we do not have an duplicates in the first case. 

900056

### 4. Droping values where boarded passangers are zero

In [12]:
Arrival_DF_0 = Arrival_DF[Arrival_DF["boarded pax"]==0.0]
pecentage0= len(Arrival_DF_0)/len(Arrival_DF)
pecentage0*100
#Arrival_DF = Arrival_DF.dropna(subset=["boarded pax"])
#Arrival_DF = Arrival_DF[Arrival_DF["boarded pax"]!=0]

0.2575395308736345

In [13]:
Arrival_DF = Arrival_DF.dropna(subset=["boarded pax"])
Arrival_DF.shape # duplicates dropped hence we are left with fewer rows .

(818992, 16)

### 5.  Pax Ratio Calculation 

I am going to find the ratio of the Pax number over the Aircraft capacity ie : 
- Pax ratio is defined as the ratio of the boarded pax to the Aircraft Capacity
$$
PA_r=Paxt/Seats
$$
where PA_r stands for Passanger Aircraft Ratio 

In [14]:
Arrival_DF.loc[:,"PAr"]=round((Arrival_DF['boarded pax']/Arrival_DF['Seat Capacity']),4)# round it to 4.sf
Arrival_DF.shape

(818992, 17)

In [15]:
greate1=Arrival_DF[(Arrival_DF["PAr"] > 1) | (Arrival_DF["PAr"] == float("inf"))] ## starting from here . 
greate1.shape

(42131, 17)

### Percentage of values greater than zero

In [16]:
greate1percent=(len(greate1)/len(Arrival_DF))*100
print("%percent of those greater than 1 is : ", greate1percent)

%percent of those greater than 1 is :  5.144250493289312


### Saving DB containing values where the pax load is greater than 1 .

In [17]:
greate1.loc[:,"Flight Number"]=greate1.loc[:,"Flight Number"].str.replace(r'\s','')
greate1.loc[:,"Flight Number"]=greate1.loc[:,"Flight Number"].apply(str)
greate1.loc[:,"Aircraft Type"]=greate1.loc[:,"Flight Number"].str[-3:] # choosing only the first right three digits . 
#value_counts()
greate1.to_excel(r"C:\Users\enock.mugabi\AOBD_Historical_Data\Investigations\PAR_greaterthan1pending.xlsx")

C:\Users\enock.mugabi\AppData\Local\Temp\ipykernel_15348\1859778492.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  greate1.loc[:,"Aircraft Type"]=greate1.loc[:,"Flight Number"].str[-3:] # choosing only the first right three digits .


In [18]:
Arrival_DF.drop(Arrival_DF[(Arrival_DF["PAr"] > 1) | (Arrival_DF["PAr"] == float("inf"))].index, inplace=True)
Arrival_DF.shape

(776861, 17)

In [19]:
Arrival_DF["boarded pax"].sum()

119404152.0

#### Extracting Month and Week Number from the the Date format 

In [20]:
Arrival_DF.loc[:,"Month"]=Arrival_DF["Date"].dt.strftime('%B')
Arrival_DF.loc[:,"WN"]=Arrival_DF["Date"].dt.isocalendar().week
Arrival_DF.loc[:,"Year"]=Arrival_DF.loc[:,"Date"].dt.year

In [21]:
Arrival_DF["Year"].unique() # Here we will use that for 2018 , 2019 and 2022 . to forsee that for 2023 . 

array([2018, 2019, 2022, 2023])

#### Chossing the most important rows For the to be trained DB

In [22]:
Arrival_DF.columns

Index(['Date', 'Day of week', 'Weekend (0/1)', 'Holiday (0/1)', 'Festival',
       'Overlap with Weekend', 'Extended weekend', 'boarded pax',
       'Original Airport', 'Destination airport', 'Arrival Datetime',
       'Seat Capacity', 'traffic_type', 'Flight Number', 'Terminal',
       'Airline Code', 'PAr', 'Month', 'WN', 'Year'],
      dtype='object')

In [23]:
Arrival_DF["traffic_type"].unique() # We have to cut the data indo traffic types .. 
Arrival_DF["traffic_type"]=Arrival_DF["traffic_type"].replace('INT','I')
Arrival_DF["traffic_type"]=Arrival_DF["traffic_type"].replace('DOM','D')

#### Dropping Airport Destinations that where considred as Null in the DB

In [24]:
Arrival_DF.dropna(subset=["Original Airport"]).copy() # Arrival_DF.dropna(subset=["Original Airport"]).copy

,Date,Day of week,Weekend (0/1),Holiday (0/1),Festival,Overlap with Weekend,Extended weekend,boarded pax,Original Airport,Destination airport,Arrival Datetime,Seat Capacity,traffic_type,Flight Number,Terminal,Airline Code,PAr,Month,WN,Year
0,2018-12-31,Monday,0,0,0,0,0,155.0,IXR,DEL,2018-12-31 00:15:00,186.0,D,G8144,T2,G8,0.8333,December,1,2018
1,2018-12-31,Monday,0,0,0,0,0,73.0,BOM,DEL,2018-12-31 22:35:00,186.0,D,G8341,T2,G8,0.3925,December,1,2018
2,2018-12-31,Monday,0,0,0,0,0,162.0,BLR,DEL,2018-12-31 23:50:00,186.0,D,G8118,T2,G8,0.8710,December,1,2018
3,2018-12-30,Sunday,1,0,0,1,1,139.0,HYD,DEL,2018-12-30 12:45:00,186.0,D,G8424,T2,G8,0.7473,December,52,2018
4,2018-12-30,Sunday,1,0,0,1,1,136.0,AMD,DEL,2018-12-30 09:50:00,186.0,D,G8720,T2,G8,0.7312,December,52,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900158,2023-12-31,Sunday,1,0,0,1,1,87.0,BOM,DEL,2023-12-31 19:10:00,232.0,D,6E882,T1,6E,0.3750,December,52,2023
900159,2023-12-31,Sunday,1,0,0,1,1,82.0,CCU,DEL,2023-12-31 22:55:00,164.0,D,UK708,T3,UK,0.5000,December,52,2023
900160,2023-12-31,Sunday,1,0,0,1,1,112.0,BLR,DEL,2023-12-31 22:20:00,188.0,D,UK818,T3,UK,0.5957,December,52,2023
900162,2023-12-31,Sunday,1,0,0,1,1,241.0,MEL,DEL,2023-12-31 16:25:00,256.0,I,AI309,T3,AI,0.9414,December,52,2023


#### Dropping classes where have PAr as Nan

In [25]:
AA=Arrival_DF.dropna(subset=["PAr"]).copy() # Arrival_DF.dropna(subset=["Original Airport"]).copy

In [26]:
AA.replace([np.inf, -np.inf], np.nan, inplace=True)

In [27]:
AA=Arrival_DF.fillna(Arrival_DF["PAr"])

In [28]:
Feature_Selection=AA 
    # This is being stored and then retrieved for feature selection 

In [29]:
%store  Feature_Selection

Stored 'Feature_Selection' (DataFrame)


In [30]:
Feature_Selection.columns

Index(['Date', 'Day of week', 'Weekend (0/1)', 'Holiday (0/1)', 'Festival',
       'Overlap with Weekend', 'Extended weekend', 'boarded pax',
       'Original Airport', 'Destination airport', 'Arrival Datetime',
       'Seat Capacity', 'traffic_type', 'Flight Number', 'Terminal',
       'Airline Code', 'PAr', 'Month', 'WN', 'Year'],
      dtype='object')